In [1]:
from sentence_transformers import CrossEncoder, SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

import faiss
import fitz
import math
import re
import torch

/Python/rag-example/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print("CUDA:", torch.cuda.is_available())
print(torch.version.cuda)

CUDA: True
12.8


In [3]:
def normalize_pdf_text(text: str) -> str:
    # Normalize line breaks
    text = re.sub(r'(?<!\n)\n(?!\n)', ' ', text)

    # Normalize multiple spaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()


# Load PDF and get texts from it
def load_pdf(path):
    doc = fitz.open(path)
    text = ''

    for page in doc:
        text += page.get_text()

    return normalize_pdf_text(text)


# Chunking with fixed token size
def chunk_fixed(text, chunk_size=300):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size):
        chunk = ' '.join(words[i: i + chunk_size])
        chunks.append(chunk)

    return chunks


# Chunking with fixed token size and overlap
def chunk_fixed_with_overlap(text, chunk_size=200, overlap=50):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size - overlap):
        chunk = ' '.join(words[i:i + chunk_size])
        chunks.append(chunk)

    return chunks


# Chunking with sentence-based
def chunk_sentence(text, max_sent=5):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks = []

    for i in range(0, len(sentences), max_sent):
        chunk = ' '.join(sentences[i:i + max_sent])
        chunks.append(chunk)

    return chunks


# Chunking with semantic-based
def chunk_semantic(text, model, n_clusters=10):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    embeddings = model.encode(sentences)

    kmeans = KMeans(n_clusters=min(n_clusters, len(sentences)))
    labels = kmeans.fit_predict(embeddings)

    clusters = {}
    for sentence, label in zip(sentences, labels):
        clusters.setdefault(label, []).append(sentence)

    return [' '.join(v) for v in clusters.values()]


# Load embedding model
model = SentenceTransformer('intfloat/multilingual-e5-base', device='cpu')

In [4]:
# Do embedding for chunks
def embed_chunks(texts):
    return model.encode([f"chunk: {t}" for t in texts])


# Do embedding for queries
def embed_queries(queries):
    return model.encode([f"query: {q}" for q in queries])


# Build FAISS index
def build_faiss_index(chunks, embeddings):
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    faiss.normalize_L2(embeddings)
    index.add(embeddings)

    return index


# Retrieval top-n
def retrieve(index, chunks, q_embed, n=20) -> list[list[str]]:
    faiss.normalize_L2(q_embed)
    D, I = index.search(q_embed, n)

    return [[chunks[i] for i in row] for row in I]

In [5]:
def normalize_text(text: str) -> str:
    return text.lower().strip()


def is_relevant_substring(gt_text: str, retrieved_chunk: str) -> bool:
    gt_norm = normalize_text(gt_text)
    chunk_norm = normalize_text(retrieved_chunk)

    return gt_norm in chunk_norm


# Recall@k with substring matching
def recall_at_k_substring(ground_truths: list[list[str]], retrieved_chunks: list[list[str]], k: int = 20) -> float:
    hits = []

    for gt_list, retrieved in zip(ground_truths, retrieved_chunks):
        found = False

        for gt in gt_list:
            for chunk in retrieved[:k]:
                if is_relevant_substring(gt, chunk):
                    found = True
                    break
            if found:
                break

        hits.append(1 if found else 0)

    return sum(hits) / len(hits)


def is_relevant_semantic(gt_text: str, chunk_text: str, threshold: float = 0.75) -> bool:
    """
    threshold:
      - 0.70-0.75 → long chunks
      - 0.80+     → sentence chunks
    """
    gt_emb = model.encode(
        [f"chunk: {gt_text}"],
        normalize_embeddings=True
    )
    chunk_emb = model.encode(
        [f"chunk: {chunk_text}"],
        normalize_embeddings=True
    )

    sim = cosine_similarity(gt_emb, chunk_emb)[0][0]

    return sim >= threshold


# Recall@k with cosine similarity
def recall_at_k_semantic(
    ground_truths: list[list[str]],
    retrieved_chunks: list[list[str]],
    k: int = 20,
    threshold: float = 0.75
):
    hits = []

    for gt_list, retrieved in zip(ground_truths, retrieved_chunks):
        found = False

        for gt in gt_list:
            for chunk in retrieved[:k]:
                if is_relevant_semantic(gt, chunk, threshold):
                    found = True
                    break
            if found:
                break

        hits.append(1 if found else 0)

    return sum(hits) / len(hits)

In [25]:
# Define constant
DATASET_PATH = 'dataset/essay_sample_2.pdf'
N = 20        # Used for retrieval top-n
K = 5         # Used for reranking top-k


# Load PDF
text = load_pdf(DATASET_PATH)


# 1. Chunking
chunks_fixed = chunk_fixed_with_overlap(text)
chunks_sentence = chunk_sentence(text)
chunks_semantic = chunk_semantic(text, model)


# 2. Embedding
emb_fixed = embed_chunks(chunks_fixed)
emb_sentence = embed_chunks(chunks_sentence)
emb_semantic = embed_chunks(chunks_semantic)


# 3. Build FAISS index
index_fixed = build_faiss_index(chunks_fixed, emb_fixed)
index_sentence = build_faiss_index(chunks_sentence, emb_sentence)
index_semantic = build_faiss_index(chunks_semantic, emb_semantic)

In [26]:
# 4. Query
queries = [
  'What are the characteristics of the oppressed?',
  'What is the dehumanization?',
  'What are the steps of the pedagogy of the oppressed?',
]

ground_truths = [
    ["oppressed", "self-depreciation"],
    ["humanity", "oppressors", "pedagogy of the oppressed", "egoistic interests"],
    ["oppressed consciousness", "oppressor consciousness", "pedagogy"]
]

q_embed = embed_queries(queries)


# 5. Retrieval
ret_fixed = retrieve(index_fixed, chunks_fixed, q_embed, N)
ret_sentence = retrieve(index_sentence, chunks_sentence, q_embed, N)
ret_semantic = retrieve(index_semantic, chunks_semantic, q_embed, N)

# Evaluation before reranking
# Using Recall@k with substring matching
print('Recall@k with substring matching')
print("Recall Fixed:", recall_at_k_substring(ground_truths, ret_fixed, N))
print("Recall Sentence:", recall_at_k_substring(ground_truths, ret_sentence, N))
print("Recall Semantic:", recall_at_k_substring(ground_truths, ret_semantic, N))

# Using Recall@k with cosine similarity
print('Recall@k with cosine similarity')
print("Recall Fixed:", recall_at_k_semantic(ground_truths, ret_fixed, N))
print("Recall Sentence:", recall_at_k_semantic(ground_truths, ret_sentence, N))
print("Recall Semantic:", recall_at_k_semantic(ground_truths, ret_semantic, N))

Recall@k with substring matching
Recall Fixed: 1.0
Recall Sentence: 1.0
Recall Semantic: 1.0
Recall@k with cosine similarity
Recall Fixed: 1.0
Recall Sentence: 1.0
Recall Semantic: 1.0


In [8]:
def relevance_vector_substring(
    retrieved: list[str],
    gt_list: list[str],
    k: int
) -> list[int]:
    rel = []

    for chunk in retrieved[:k]:
        hit = any(is_relevant_substring(gt, chunk) for gt in gt_list)
        rel.append(1 if hit else 0)

    return rel


def relevance_vector_semantic(
    retrieved: list[str],
    gt_list: list[str],
    k: int,
    threshold: float = 0.75
) -> list[int]:
    rel = []

    for chunk in retrieved[:k]:
        hit = any(
            is_relevant_semantic(gt, chunk, threshold)
            for gt in gt_list
        )
        rel.append(1 if hit else 0)

    return rel


def mean_reciprocal_rank(
    retrieved_chunks: list[list[str]],
    ground_truths: list[list[str]],
    k: int,
    relevance_fn
) -> float:
    scores = []

    for retrieved, gt_list in zip(retrieved_chunks, ground_truths):
        rel = relevance_fn(retrieved, gt_list, k)

        rr = 0.0
        for idx, r in enumerate(rel):
            if r == 1:
                rr = 1.0 / (idx + 1)
                break

        scores.append(rr)

    return sum(scores) / len(scores)


def average_precision(rel: list[int]) -> float:
    if sum(rel) == 0:
        return 0.0

    score = 0.0
    hits = 0

    for i, r in enumerate(rel):
        if r == 1:
            hits += 1
            score += hits / (i + 1)

    return score / sum(rel)


def mean_average_precision(
    retrieved_chunks: list[list[str]],
    ground_truths: list[list[str]],
    k: int,
    relevance_fn
) -> float:
    ap_scores = []

    for retrieved, gt_list in zip(retrieved_chunks, ground_truths):
        rel = relevance_fn(retrieved, gt_list, k)
        ap_scores.append(average_precision(rel))

    return sum(ap_scores) / len(ap_scores)


def dcg(rel: list[int]) -> float:
    return sum(
        r / math.log2(i + 2)
        for i, r in enumerate(rel)
    )


def ndcg_at_k(
    retrieved_chunks: list[list[str]],
    ground_truths: list[list[str]],
    k: int,
    relevance_fn
) -> float:
    scores = []

    for retrieved, gt_list in zip(retrieved_chunks, ground_truths):
        rel = relevance_fn(retrieved, gt_list, k)

        ideal_rel = sorted(rel, reverse=True)

        dcg_val = dcg(rel)
        idcg_val = dcg(ideal_rel)

        scores.append(dcg_val / idcg_val if idcg_val > 0 else 0.0)

    return sum(scores) / len(scores)

In [27]:
# Evaluation before reranking
print('Fixed')
print('MRR: ', mean_reciprocal_rank(ret_fixed, ground_truths, N, relevance_vector_substring))
print('MAP: ', mean_average_precision(ret_fixed, ground_truths, N, relevance_vector_substring))
print('NDCG: ', ndcg_at_k(ret_fixed, ground_truths, N, relevance_vector_substring))

print('Sentence')
print('MRR: ', mean_reciprocal_rank(ret_sentence, ground_truths, N, relevance_vector_substring))
print('MAP: ', mean_average_precision(ret_sentence, ground_truths, N, relevance_vector_substring))
print('NDCG: ', ndcg_at_k(ret_sentence, ground_truths, N, relevance_vector_substring))

print('Semantic')
print('MRR: ', mean_reciprocal_rank(ret_semantic, ground_truths, N, relevance_vector_semantic))
print('MAP: ', mean_average_precision(ret_semantic, ground_truths, N, relevance_vector_semantic))
print('NDCG: ', ndcg_at_k(ret_semantic, ground_truths, N, relevance_vector_semantic))

Fixed
MRR:  1.0
MAP:  0.9676958242181479
NDCG:  0.9899860230036909
Sentence
MRR:  1.0
MAP:  0.9538653084471905
NDCG:  0.9877579300323126
Semantic
MRR:  1.0
MAP:  1.0
NDCG:  1.0


In [10]:
reranker = CrossEncoder(
    'BAAI/bge-reranker-base',
    device='cpu'
)


def rerank_chunks(queries: list[str], retrieved_chunks: list[list[str]], top_k: int = 5) -> list[list[str]]:
    reranked_list = []

    for query, retrieved in zip(queries, retrieved_chunks):
        pairs = [(query, chunk) for chunk in retrieved]

        scores = reranker.predict(pairs)

        ranked = sorted(
            zip(retrieved, scores),
            key=lambda x: x[1],
            reverse=True
        )

        reranked_chunks_only = [chunk for chunk, _ in ranked]

        reranked_list.append(reranked_chunks_only[:top_k])

    return reranked_list

In [28]:
reranked_fixed = rerank_chunks(queries, ret_fixed, K)
reranked_sentence = rerank_chunks(queries, ret_sentence, K)
reranked_semantic = rerank_chunks(queries, ret_semantic, K)

# Evaluation after reranking
print('Fixed')
print('MRR: ', mean_reciprocal_rank(reranked_fixed, ground_truths, K, relevance_vector_substring))
print('MAP: ', mean_average_precision(reranked_fixed, ground_truths, K, relevance_vector_substring))
print('NDCG: ', ndcg_at_k(reranked_fixed, ground_truths, K, relevance_vector_substring))

print('Sentence')
print('MRR: ', mean_reciprocal_rank(reranked_sentence, ground_truths, K, relevance_vector_substring))
print('MAP: ', mean_average_precision(reranked_sentence, ground_truths, K, relevance_vector_substring))
print('NDCG: ', ndcg_at_k(reranked_sentence, ground_truths, K, relevance_vector_substring))

print('Semantic')
print('MRR: ', mean_reciprocal_rank(reranked_semantic, ground_truths, K, relevance_vector_semantic))
print('MAP: ', mean_average_precision(reranked_semantic, ground_truths, K, relevance_vector_semantic))
print('NDCG: ', ndcg_at_k(reranked_semantic, ground_truths, K, relevance_vector_semantic))

Fixed
MRR:  1.0
MAP:  1.0
NDCG:  1.0
Sentence
MRR:  1.0
MAP:  1.0
NDCG:  1.0
Semantic
MRR:  1.0
MAP:  1.0
NDCG:  1.0


In [12]:
# MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

# Define pretrained tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Define pretrained LLM
torch.cuda.empty_cache()
model_llm = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='cuda',
    dtype=torch.float16,
)

In [13]:
print("GPU:", torch.cuda.get_device_name(0))
print("Device:", next(model_llm.parameters()).device)

GPU: NVIDIA GeForce RTX 3050 Laptop GPU
Device: cuda:0


In [23]:
def select_context(chunks: list[str], max_chunks: int = 5) -> str:
    return "\n\n".join(chunks[:max_chunks])


def build_prompt(query: str, context: str) -> str:
    return f"""\
Anda adalah sistem tanya jawab berbasis dokumen.

Gunakan informasi di bawah ini untuk menjawab pertanyaan.
Jawaban harus berupa ringkasan langsung dari konteks.
Jangan menambahkan konsep, istilah, atau penjelasan yang tidak tertulis secara eksplisit di konteks.
Jawab dalam 2–3 kalimat naratif, bukan poin-poin.
Jika jawabannya tidak ada di konteks, jawab: "Informasi tidak ditemukan dalam dokumen."

### KONTEKS
{context}

### PERTANYAAN
{query}

### JAWABAN
"""


def generate_answer(prompt: str, max_new_tokens: int = 256) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(model_llm.device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output = model_llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.2,
            top_p=0.9,
            repetition_penalty=1.25,
            no_repeat_ngram_size=4,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = output[0][input_len:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )


def rag_answer(
    query: str,
    index,
    chunks,
    n: int = 20,
    k: int = 5,
    used_reranker: bool = True
) -> str:
    MAXIMUM_CONTEXT_CHUNKS = 2

    # Embed query
    q_embed = embed_queries([query])

    # Retrieval
    retrieved = retrieve(index, chunks, q_embed, n)[0]

    # Reranking
    reranked = rerank_chunks([query], [retrieved], k)[0]

    # Context selection
    context = select_context(reranked if used_reranker else retrieved, MAXIMUM_CONTEXT_CHUNKS)
    print(context)

    # Prompt
    prompt = build_prompt(query, context)

    # LLM generation
    answer = generate_answer(prompt)

    return answer

In [29]:
query = "What are the steps of the pedagogy of the oppressed?"

# Without reranker
answer = rag_answer(
    query=query,
    index=index_semantic,
    chunks=chunks_semantic,
    n=N,
    k=K,
    used_reranker=False
)

print(answer)

This, then, is the great humanistic and historical task of the op­ pressed: to liberate themselves and their oppressors as well. Only power that springs from the weakness of the oppressed will be sufficiently strong to free both. In order to have the continued opportunity to express their "generosity," the oppressors must perpetuate injustice as well. This lesson and this apprenticeship must come, however, from the oppressed themselves and from those who are truly solidary with them. As individuals or as peoples, by fighting for the restoration of their humanity they will be attempting the restoration of true generosity. This is their model of humanity. In this example, the overseer, in order to make sure of his job, must be as tough as the owner—and more so. It is rather the indispensable condition for the quest for human com­ pletion. To surmount the situation of oppression, people must first criti­ cally recognize its causes, so that through transforming action they can create a new

In [30]:
# With reranker
answer = rag_answer(
    query=query,
    index=index_semantic,
    chunks=chunks_semantic,
    n=N,
    k=K
)

print(answer)

PEDAGOGY of the OPPRESSED CHAPTER 1 W hile the problem of humanization has always, from an axiological point of view, been humankind's central problem, it now takes on the character of an inescapable concern.l Concern for humanization leads at once to the recognition of dehumanization, not only as an ontological possibility but as an historical reality And as an individual perceives the extent of dehu- manization, he or she rtiay ask if humanization is a viable possibility. The current movements of rebellion, especially those of youth, while they necessarily reflect the peculiarities of their respective settings, manifest in their essence this preoccupation with people as beings in the world and with the world— preoccupation with what and how they are "being." As they place consumer civiliza­ tion in judgment, denounce bureaucracies of all types, demand the transformation of the universities (changing the rigid nature of the teacher-student relationship and placing that relationship wi